# Entraînement de Yolo sur un echantillon du fonds Thierry (images avec annotations)

1) Division train-val
2) Resize à 640x640 pixel
3) Installation de Yolo
4) Entraînement de Yolo
5) Test
6) Evaluation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip install pillow

## Split dataset + resize images (640×640)

In [ ]:
import os
import random
import shutil
from PIL import Image

# ================= CONFIG =================
SOURCE_DIR = "/content/drive/MyDrive/images"
OUTPUT_DIR = "/content/drive/MyDrive/Fonds Thierry/Méthode supervisé - Yolo/Dataset et model training tag_1/Benchmark_5/dataset"
TRAIN_RATIO = 0.8
IMG_SIZE = (640, 640)
SEED = 42
# ==========================================

random.seed(SEED)

# Nettoyage du dossier output s'il existe
def prepare_dirs():
    for split in ["train", "val"]:
        os.makedirs(os.path.join(OUTPUT_DIR, split), exist_ok=True)

# Fonction pour réduire la taille des images et les enregistrer
def resize_and_save(src_path, dst_path):
    with Image.open(src_path) as img:
        img = img.convert("RGB")
        img = img.resize(IMG_SIZE, Image.BILINEAR)
        img.save(dst_path, quality=95)

prepare_dirs()

# Division des images en train/val pour chaque tag
for tag in os.listdir(SOURCE_DIR):
    tag_path = os.path.join(SOURCE_DIR, tag)
    if not os.path.isdir(tag_path):
        continue

    images = [
        f for f in os.listdir(tag_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    random.shuffle(images)
    split_idx = int(len(images) * TRAIN_RATIO)

    splits = {
        "train": images[:split_idx],
        "val": images[split_idx:]
    }

    for split, imgs in splits.items():
        out_dir = os.path.join(OUTPUT_DIR, split, tag)
        os.makedirs(out_dir, exist_ok=True)

        for img_name in imgs:
            src = os.path.join(tag_path, img_name)
            dst = os.path.join(out_dir, img_name)

            resize_and_save(src, dst)

    print(f"Classe '{tag}' → {len(images)} images")

print("Dataset prêt pour YOLO")


Classe '12 Reproduction de plan architectural' → 97 images
Classe '13 Autre document reproduit gravure dessin etc' → 105 images
Classe '14 Matériel de conditionnement' → 92 images
Classe '15 Autre' → 3 images
Classe '11 Photographie' → 162 images
Dataset prêt pour YOLO


## Training YOLOv11

In [ ]:
%pip install ultralytics

In [ ]:
from ultralytics import YOLO

# Modèle conseillé avec GPU + 5 classes
model = YOLO("yolo11n-cls.pt")

model.train(
    data="/content/drive/MyDrive/Fonds Thierry/Méthode supervisé - Yolo/Dataset et model training tag_1/Benchmark_5/dataset",      # doit contenir train/ et val/
    epochs=60,
    imgsz=640,           # cohérent avec resize
    batch=16,            # abbassa a 16 se OOM
    device=0,            # GPU
    workers=8,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    dropout=0.2,
    patience=10,         # early stopping
    amp=True             # mixed precision (consigliato su GPU)
)


Ultralytics 8.3.249 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Fonds Thierry/Méthode supervisé - Yolo/Dataset et model training tag_1/Benchmark_5/dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=Fa

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x799a40188d40>
curves: []
curves_results: []
fitness: 0.936170220375061
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8723404407501221, 'metrics/accuracy_top5': 1.0, 'fitness': 0.936170220375061}
save_dir: PosixPath('/content/runs/classify/train')
speed: {'preprocess': 0.5660160106364198, 'inference': 1.8897830744769655, 'loss': 0.0005861382963670853, 'postprocess': 0.0007490212699346383}
task: 'classify'
top1: 0.8723404407501221
top5: 1.0

## Test du modèle

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/classify/train/weights/best.pt")

results = model("test.jpg")

for r in results:
    cls_id = r.probs.top1
    print(r.names[cls_id], r.probs.top1conf)
